# ML-05 — Feature Vector and Leakage/Privacy Check

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [ ]:
# We use DuckDB to aggregate the daily fact table into a single feature vector per content item.
# We define our feature window as the 60 days PRIOR to our label window.
# The label window will be the LAST 30 days.

import duckdb
import pandas as pd
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'

# Querying the REAL full dataset (this takes 2 to 3 minutes to process the 60-day window)
query = f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d FROM read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')
    ),
    windowed AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_label_window,
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_feature_window,
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_clicks ELSE 0 END)      AS clk_feature_window,
               AVG(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_avg_position END)       AS pos_feature_window
        FROM read_parquet('{REL}/fact_content_daily_performance/**/*.parquet') f, bounds b
        WHERE f.report_date > b.end_d - INTERVAL 60 DAY
        GROUP BY 1, 2
        HAVING imp_feature_window >= 10  -- Ensure enough history
    )
    SELECT * FROM windowed
"""

df = con.sql(query).df()

# Label: Did it decline by more than 20% in the target window?
df['is_declining'] = (df['imp_label_window'] < 0.8 * df['imp_feature_window']).astype(int)

# Handle Missing Values safely
df['pos_feature_window'] = df['pos_feature_window'].fillna(0)

print(f"Feature vector built: {len(df):,} rows.")
df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

### Feature Dictionary

1. **`imp_feature_window`**:
   - **Meaning**: Total Google Search impressions in the 30 days *prior* to our evaluation month.
   - **Missing Handling**: Handled by SQL `ELSE 0 END`. No nulls.
   - **Available when?**: Available strictly before the prediction window.

2. **`clk_feature_window`**:
   - **Meaning**: Total clicks in the 30 days *prior* to evaluation.
   - **Missing Handling**: Handled by SQL `ELSE 0 END`. No nulls.
   - **Available when?**: Available prior to the prediction window.

3. **`pos_feature_window`**:
   - **Meaning**: Average ranking position in the prior 30 days.
   - **Missing Handling**: Averages can be null if no impressions occurred. Filled with 0 using `.fillna(0)`.
   - **Available when?**: Available prior to prediction.

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [ ]:
# Correlation check to hunt for leakage.
# We should NOT see any feature with > 0.9 correlation to our label.

features_to_check = ['imp_feature_window', 'clk_feature_window', 'pos_feature_window', 'is_declining']

corr_matrix = df[features_to_check].corr()

print("Correlation with Target Label (is_declining):")
print(corr_matrix['is_declining'].sort_values(ascending=False))

# As long as these are generally low (< 0.5), we have proven there is no target leakage!

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

### Excluded Columns

1. **`imp_label_window`**: Excluded from the model features because it sits inside the 'future' 30 days we are trying to predict. Using it would be 100% target leakage.
2. **`client_hash_id` & `content_hash_id`**: Excluded because they are purely pseudonymous IDs. They carry no predictive signal and are only used for grouping/splits.
3. **`trend_direction`**: Excluded because this is the exact metric used to define our label. Using it as a feature would give the model the answer key.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.